# LoRA Fine-tuning for EN → PT Translation

Study project — fine-tuning a pre-trained translation model with LoRA.

Steps:
1. Install dependencies & verify GPU
2. Load and normalize the dataset
3. Load model + tokenize
4. Configure LoRA
5. Train
6. Save adapter
7. Test and compare

## 1. Install & GPU check

In [ ]:
!pip install -q transformers datasets sentencepiece sacrebleu accelerate evaluate peft
!pip install -q --upgrade torchao

In [ ]:
# Colab: after the installs above, go to Runtime → Restart session, then run from the next cell
# (the upgrade of torchao only takes effect after a restart)

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no GPU found — training will be very slow on CPU')

## 2. Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('VanessaSchenkel/translation-en-pt', field='data')
print(dataset)
print('\nColumns:', dataset['train'].column_names)
print('\nFirst example:', dataset['train'][0])

In [ ]:
def normalize_dataset(dataset):
    """If the 'translation' column is a dict {en: ..., pt: ...}, expands it to flat columns."""
    cols = dataset['train'].column_names
    if 'translation' not in cols:
        return dataset

    sample = dataset['train'][0]['translation']
    if not isinstance(sample, dict):
        return dataset

    lang_keys = list(sample.keys())
    en_key = next((k for k in ['en', 'english'] if k in lang_keys), None)
    pt_key = next((k for k in ['pt', 'portuguese'] if k in lang_keys), None)

    if not en_key or not pt_key:
        raise ValueError(f"Column 'translation' has keys {lang_keys}. Expected 'en' and 'pt'.")

    print(f"Expanding 'translation.{en_key}' and 'translation.{pt_key}' → flat columns 'en' and 'pt'")

    def flatten(batch):
        return {
            'en': [t[en_key] for t in batch['translation']],
            'pt': [t[pt_key] for t in batch['translation']],
        }

    return dataset.map(flatten, batched=True, remove_columns=cols, desc='Normalizing')

dataset = normalize_dataset(dataset)
print('Columns after normalization:', dataset['train'].column_names)

In [ ]:
import pandas as pd
pd.DataFrame(dataset['train'][:5])

## 3. Load model + tokenize

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = 'Helsinki-NLP/opus-mt-en-ROMANCE'
MAX_LEN = 128
TRAIN_SUBSET = 0.05  # use 5% for quick runs — set to None for full training

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
base_model = MarianMTModel.from_pretrained(MODEL_NAME)

print('Model loaded!')
print(f'Total parameters: {base_model.num_parameters():,}')

In [ ]:
# Optionally limit the dataset size for faster iteration
if TRAIN_SUBSET is not None:
    n = int(len(dataset['train']) * TRAIN_SUBSET)
    dataset['train'] = dataset['train'].select(range(n))
    print(f'Using {TRAIN_SUBSET*100:.0f}% of dataset → {n} examples')

def preprocess(examples):
    inputs = [f'>>pt<< {t}' for t in examples['en']]
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True, padding=False)
    labels = tokenizer(text_target=examples['pt'], max_length=MAX_LEN, truncation=True, padding=False)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = dataset.map(
    preprocess, batched=True,
    remove_columns=dataset['train'].column_names,
    desc='Tokenizing'
)

if 'validation' not in tokenized:
    split = tokenized['train'].train_test_split(test_size=0.05, seed=42)
    tokenized['train'] = split['train']
    tokenized['validation'] = split['test']

print(f"Train: {len(tokenized['train'])} | Validation: {len(tokenized['validation'])}")

## 4. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,           # rank: size of the trainable matrices inside each attention layer
    lora_alpha=32,  # scale factor (convention: 2 × r)
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],  # attention layers where LoRA is applied
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~884k out of ~78M — only ~1% updated

## 5. Train

In [ ]:
import evaluate
import numpy as np
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)

bleu = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[[l.strip()] for l in decoded_labels]
    )
    return {'bleu': round(result['score'], 2)}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    predict_with_generate=True,
    generation_max_length=128,
    fp16=torch.cuda.is_available(),  # only use fp16 if GPU is available
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6. Save adapter

In [ ]:
import os

ADAPTER_DIR = './outputs/adapter-en-pt'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f'Adapter saved to: {ADAPTER_DIR}')
print(f'Total size: {size_mb:.1f} MB  (vs ~300 MB for a full model copy)')

## 7. Test and compare

In [ ]:
from peft import PeftModel

def translate_with(texts, tok, mdl):
    inputs = [f'>>pt<< {t}' for t in texts]
    enc = tok(inputs, return_tensors='pt', padding=True, truncation=True, max_length=128)
    gen_model = mdl.base_model if isinstance(mdl, PeftModel) else mdl
    gen = gen_model.generate(**enc, num_beams=4, max_length=128)
    return tok.batch_decode(gen, skip_special_tokens=True)

base_only = MarianMTModel.from_pretrained(MODEL_NAME)

sentences = [
    'Hello, how are you?',
    'The weather is beautiful today.',
    'I love learning new languages.',
    'Artificial intelligence is changing the world.',
]

print('=== Base model (no fine-tuning) ===')
for s, t in zip(sentences, translate_with(sentences, tokenizer, base_only)):
    print(f'  EN: {s}')
    print(f'  PT: {t}\n')

print('=== Model with LoRA adapter ===')
for s, t in zip(sentences, translate_with(sentences, tokenizer, model)):
    print(f'  EN: {s}')
    print(f'  PT: {t}\n')